In [ ]:
import scanpy as sc
import pandas as pd
import plotnine as gg
import matplotlib.pyplot as plt
from utils import load_fitness_data

from essential.utils import PLOTNINE_DEFAULT_THEME_2

plt.rcParams["svg.fonttype"] = "none"


ADATA_PATH = (
    "/workspace/experiments/04302026_analysis_data_v2/adata_de122_lce75_merged.h5ad"
)
MODULE_INFO_PATH = "/workspace/experiments/04152026_kegg/module_info.csv"
MODULE_PREDICTION_PATH = "/workspace/experiments/04152026_kegg/module_prediction.json"
MIN_GENES = 2

### transcriptomics data properties

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(adata)
print(adata.obs["target"].unique())
print(adata.obs["target"].value_counts().mean())
print(adata.obs["target"].value_counts().median())

In [ ]:
# adata = adata[adata.obs["experiment"] == "lce75"]
# adata = adata[adata.obs["experiment"] == "de122"]
adata

### individual modules

In [ ]:
def compute_de(adata, genes, n_genes=10):
    adata_sub = adata[adata.obs["target"].isin(genes)].copy()
    sc.tl.rank_genes_groups(
        adata_sub, groupby="target", method="wilcoxon", reference="rest"
    )

    de_dfs = {}
    marker_genes = []
    for cls in genes:
        df = sc.get.rank_genes_groups_df(adata_sub, group=cls).set_index("names")
        de_dfs[cls] = df
        marker_genes.append(df.head(n_genes).index.tolist())
    return de_dfs, marker_genes


def compute_de_single(adata, gene, n_genes=10):
    adata_sub = adata[adata.obs["target"].isin([gene, "nontargeting"])].copy()
    sc.tl.rank_genes_groups(
        adata_sub, groupby="target", method="wilcoxon", reference="rest"
    )

    return sc.get.rank_genes_groups_df(adata_sub, group=gene).set_index("names")

In [ ]:
de_dfs, _ = compute_de(adata, ["dut", "tmk"])
genes1 = de_dfs["dut"].head(5)
genes2 = de_dfs["tmk"].head(5)
genes1 = genes1.index.tolist()
genes2 = genes2.index.tolist()

# df1 = compute_de(adata, ["dut", "nontargeting"])[0]["dut"].head(5)
# df2 = compute_de(adata, ["tmk", "nontargeting"])[0]["tmk"].head(5)
# genes1 = df1.index.tolist()
# genes2 = df2.index.tolist()

In [ ]:
%matplotlib inline

In [ ]:
# genes_to_plot = genes1 + genes2

genes_to_plot = [
    "ung",  # uracil-DNA glycosylase — direct U-in-DNA sensor (dut anchor; note: largely constitutive)
    "nfo",  # AP endonuclease (EndoIV) — inducible, dut-leaning BER readout
    "recA",  # SOS master regulator (shared)
    "lexA",  # SOS repressor (shared)
    "sulA",  # strongest classic SOS reporter (shared)
    "recN",  # DSB repair — dut futile-cycle product
    "ruvB",  # Holliday-junction processing — dut DSB/recombination
    "umuC",  # TLS Pol V, SOS (shared)
    "priA",  # replication-restart primosome — tmk stalled-fork contrast
    "recF",  # ssDNA-gap repair — tmk fork contrast
]
genes = ["dut", "tmk", "nontargeting"]
adata_sub = adata[adata.obs["target"].isin(genes)].copy()
n_per_group = (
    adata_sub.obs[adata_sub.obs["target"] != "nontargeting"]["target"]
    .value_counts()
    .loc[lambda x: x >= 2]
    .min()
)
nt_idx = (
    adata_sub.obs[adata_sub.obs["target"] == "nontargeting"]
    .sample(n=n_per_group, random_state=0)
    .index
)
keep_idx = (
    adata_sub.obs[adata_sub.obs["target"] != "nontargeting"].index.tolist()
    + nt_idx.tolist()
)
adata_sub = adata_sub[keep_idx].copy()

sc.pl.rank_de_genes(adata_sub, var_names=genes_to_plot, groupby="target")

de_df, markers = compute_de(adata, genes)

In [ ]:
res = compute_de_single(adata, "dut").head(25)
print("dut vs nontargeting:")
print(", ".join(res.index.tolist()))
res

In [ ]:
res = compute_de_single(adata, "tmk").head(25)
print("tmk vs nontargeting:")
print(", ".join(res.index.tolist()))

In [ ]:
%matplotlib inline

In [ ]:
adata_sub = adata[adata.obs["target"].isin(genes)].copy()
sc.tl.rank_genes_groups(
    adata_sub, groupby="target", method="wilcoxon", reference="rest"
)
sc.pl.rank_genes_groups_dotplot(adata_sub)
plt.show()

In [ ]:
markers[0]

### pathways properties

In [ ]:
module_info = pd.read_csv(MODULE_INFO_PATH)
module_prediction = pd.read_json(MODULE_PREDICTION_PATH)
fitness_data = load_fitness_data()

module_df = module_info.merge(module_prediction, on="module_id", how="left").query(
    "n_genes >= @MIN_GENES"
)
# .assign(activity_prediction=lambda x: x["activity_prediction"].replace("partially active", "inactive"))
equivalence_class_df = pd.read_csv("module_equivalence_results.csv")

n_eq_classes = (
    equivalence_class_df.groupby("module_id")["equivalence_class"]
    .nunique()
    .to_frame("n_equivalence_classes")
)
module_df = module_df.merge(n_eq_classes, left_on="module_id", right_index=True)

In [ ]:
fig = (
    gg.ggplot(module_df, gg.aes(x="n_equivalence_classes"))
    + gg.geom_histogram(binwidth=1, fill="#2596be", color="black", size=0.1)
    + gg.scale_x_continuous(breaks=range(10))
    + PLOTNINE_DEFAULT_THEME_2
    + gg.theme(figure_size=(3.5, 1.5))
    + gg.labs(x="# of equivalence classes", y="frequency")
    + gg.facet_wrap("activity_prediction", scales="free_y")
    + gg.scale_y_continuous(expand=(0, 0), limits=(0, 20))
)
fig.save("n_equivalence_classes.svg")
fig

In [ ]:
module_df

In [ ]:
print((module_df["n_equivalence_classes"] >= 2).sum())
print((module_df["n_equivalence_classes"] >= 2).mean())

In [ ]:
fitness_data

In [ ]:
equivalence_class_df["perturbation"].nunique()

In [ ]:
gene_props = (
    equivalence_class_df.groupby("perturbation")
    .apply(lambda x: "no phenotype" not in x["equivalence_class"].values)
    .replace({False: "gene in control eq. class", True: "gene in phenotype eq. class"})
    .to_frame("has_phenotype")
    .merge(fitness_data, left_index=True, right_on="gene")
)
gene_props

In [ ]:
fig = (
    gg.ggplot(gene_props, gg.aes(x="T3", y="..density..", fill="has_phenotype"))
    + gg.geom_histogram(binwidth=0.5, alpha=0.5, position="identity")
    + PLOTNINE_DEFAULT_THEME_2
    + gg.labs(fill="", x="fitness (T+24h)")
    + gg.theme(
        figure_size=(3, 1.5),
    )
    + gg.scale_y_continuous(expand=(0, 0))
    + gg.scale_fill_manual(
        values={
            "gene in control eq. class": "#555555",
            "gene in phenotype eq. class": "#2E86AB",
        }
    )
)
fig.save("eq_class_vs_fitness.svg")

In [ ]:
gene_props

In [ ]:
genes_with_phenotypes = gene_props.query("has_phenotype == 'gene in phenotype eq. class'")
genes_with_phenotypes

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)

In [ ]:
adata

In [ ]:
for gene in genes_with_phenotypes["gene"]:
    print(gene)

In [ ]:
%matplotlib inline

In [ ]:
adata_subset

In [ ]:
genes_with_toxic_subtrates = [
    "dut",
    "thyA",
    "fbaA",
    "gapA",
    "gnd",
    "hemB",
    "hemD",
    "hemE",
    "nadC",
    "ackA",
    "lpxC",
    "mutT",
    "rdgB",
    "gloA",
    "frmA",
    "nudC"
]

In [ ]:
adata.obs["substrate_toxicity"] = (
    adata.obs["target"]
    .isin(genes_with_toxic_subtrates)
    .replace({True: "toxic", False: "unknown"})
)

In [ ]:
equivalence_class_df.loc[lambda x: x["perturbation"].isin(genes_with_toxic_subtrates)].sort_values("module_id")

In [ ]:
adata_subset.X[0].max()

In [ ]:
adata_subset = adata[adata.obs["target"].isin(genes_with_phenotypes["gene"])].copy()

sc.pp.neighbors(adata_subset, use_rep="scvi_latent", n_neighbors=15)

# sc.pp.highly_variable_genes(adata_subset, n_top_genes=500, flavor="seurat_v3")
# adata_subset = adata_subset[:, adata_subset.var["highly_variable"]].copy()
# sc.pp.normalize_total(adata_subset)
# sc.pp.log1p(adata_subset)
# sc.pp.pca(adata_subset, n_comps=50)
# sc.pp.neighbors(adata_subset, use_rep="X_pca", n_neighbors=15)

sc.tl.umap(adata_subset, min_dist=0.4)

adata_subset.obs["UMAP1"] = adata_subset.obsm["X_umap"][:, 0]
adata_subset.obs["UMAP2"] = adata_subset.obsm["X_umap"][:, 1]
sc.pl.umap(adata_subset, color="substrate_toxicity")

In [ ]:
import plotly.express as px

plot_df = adata_subset.obs.copy()
fig = px.scatter(
    plot_df,
    x="UMAP1",
    y="UMAP2",
    color="substrate_toxicity",
    hover_data=["target"],
    title="Interactive UMAP: Annotated Leiden Case",
)
fig.update_traces(marker=dict(size=3))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
equivalence_class_df.query("perturbation == 'proA'")